In [2]:
!nvidia-smi

Fri Sep 18 20:28:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install -q mlflow scikit-learn joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
import json
import hashlib
import shutil
import joblib
import mlflow

import numpy as np

from datetime import datetime

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [14]:
# Carrega o dataset
X, y = load_breast_cancer(return_X_y=True)

# Divide os dados
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Cria o modelo
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Treina
model.fit(X_train, y_train)

# Predição
predictions = model.predict(X_test)

# Métricas
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy:", round(accuracy, 4))
print("F1-Score:", round(f1, 4))

Accuracy: 0.958
F1-Score: 0.967


In [15]:
mlflow.set_experiment("MLOPS_WORKSHOP")

with mlflow.start_run() as run:

    mlflow.log_param("algorithm", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)

    run_id = run.info.run_id

print("RUN ID:", run_id)

2026/09/18 20:31:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/18 20:31:59 INFO mlflow.store.db.utils: Updating database tables
2026/09/18 20:32:01 INFO mlflow.tracking.fluent: Experiment with name 'MLOPS_WORKSHOP' does not exist. Creating a new experiment.


RUN ID: 84370a0f59f84719bfeda71cfb92d999


In [16]:
os.makedirs("models", exist_ok=True)

MODEL_VERSION = "v1"

model_path = f"models/model_{MODEL_VERSION}.pkl"

joblib.dump(
    model,
    model_path
)

print("Modelo criado:")
print(model_path)

Modelo criado:
models/model_v1.pkl


In [17]:
def sha256_file(filename):

    sha256 = hashlib.sha256()

    with open(filename, "rb") as file:

        for block in iter(
            lambda: file.read(4096),
            b""
        ):
            sha256.update(block)

    return sha256.hexdigest()


model_hash = sha256_file(model_path)

print("Modelo:", model_path)
print("SHA-256:", model_hash)

Modelo: models/model_v1.pkl
SHA-256: 5911d92693f71afafbd54a16834a822b1fb49cb9049535444241fb1770e3d02e


In [19]:
MIN_ACCURACY = 0.93
MIN_F1 = 0.93

quality_gate = (
    accuracy >= MIN_ACCURACY
    and
    f1 >= MIN_F1
)

print("Accuracy:", round(accuracy, 4))
print("F1:", round(f1, 4))

if quality_gate:
    print("QUALITY GATE: PASS")
else:
    print("QUALITY GATE: FAIL")

Accuracy: 0.958
F1: 0.967
QUALITY GATE: PASS


In [21]:
os.makedirs("registry", exist_ok=True)

if quality_gate:
    registry_path = f"registry/model_{MODEL_VERSION}.pkl"

    shutil.copy(
        model_path,
        registry_path
    )

    print("PROMOÇÃO: APROVADA")
    print("MODEL REGISTRY:", registry_path)
else:
    print("PROMOÇÃO: BLOQUEADA")

PROMOÇÃO: APROVADA
MODEL REGISTRY: registry/model_v1.pkl


In [22]:
metadata = {

    "model_version": MODEL_VERSION,

    "algorithm": "RandomForestClassifier",

    "accuracy": round(accuracy, 4),

    "f1_score": round(f1, 4),

    "sha256": model_hash,

    "mlflow_run_id": run_id,

    "created_at": datetime.now().isoformat()
}

with open(
    "registry/model_metadata.json",
    "w"
) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )

print(json.dumps(metadata, indent=4))

{
    "model_version": "v1",
    "algorithm": "RandomForestClassifier",
    "accuracy": 0.958,
    "f1_score": 0.967,
    "sha256": "5911d92693f71afafbd54a16834a822b1fb49cb9049535444241fb1770e3d02e",
    "mlflow_run_id": "84370a0f59f84719bfeda71cfb92d999",
    "created_at": "2026-09-18T20:40:57.254753"
}


In [23]:
baseline_accuracy = accuracy

print(
    "Baseline Accuracy:",
    round(baseline_accuracy, 4)
)

Baseline Accuracy: 0.958


In [24]:
X_production = X_test * 1.25

production_predictions = model.predict(
    X_production
)

production_accuracy = accuracy_score(
    y_test,
    production_predictions
)

print(
    "Accuracy baseline:",
    round(baseline_accuracy, 4)
)

print(
    "Accuracy produção:",
    round(production_accuracy, 4)
)

Accuracy baseline: 0.958
Accuracy produção: 0.7273


In [27]:
DRIFT_THRESHOLD = 0.10

performance_drop = (baseline_accuracy - production_accuracy)

print(
    "Degradação:",
    round(performance_drop, 4)
)

if performance_drop > DRIFT_THRESHOLD:
    drift_detected = True
    print("ALERTA: DRIFT DETECTADO")
else:
    drift_detected = False
    print("MODELO ESTÁVEL")

Degradação: 0.2308
ALERTA: DRIFT DETECTADO


In [30]:
if drift_detected:
    print("CONTINUOUS TRAINING: TRIGGERED")

    model_v2 = RandomForestClassifier(
        n_estimators=150,
        random_state=42
    )

    model_v2.fit(
        X_train,
        y_train
    )

    predictions_v2 = model_v2.predict(
        X_test
    )

    accuracy_v2 = accuracy_score(
        y_test,
        predictions_v2
    )

    print("Nova versão: v2")

    print(
        "Accuracy v2:",
        round(accuracy_v2, 4)
    )

CONTINUOUS TRAINING: TRIGGERED
Nova versão: v2
Accuracy v2: 0.958


In [31]:
print("""
====================================================
          PIPELINE MLOPS — RESULTADO FINAL
====================================================

DADOS
  |
  v
TREINAMENTO
  |
  v
AVALIAÇÃO
  |
  v
MLFLOW TRACKING
  |
  v
VERSIONAMENTO
  |
  v
QUALITY GATE
  |
  v
MODEL REGISTRY
  |
  v
MONITORAMENTO
  |
  v
DRIFT
  |
  v
CONTINUOUS TRAINING

====================================================
""")


          PIPELINE MLOPS — RESULTADO FINAL

DADOS
  |
  v
TREINAMENTO
  |
  v
AVALIAÇÃO
  |
  v
MLFLOW TRACKING
  |
  v
VERSIONAMENTO
  |
  v
QUALITY GATE
  |
  v
MODEL REGISTRY
  |
  v
MONITORAMENTO
  |
  v
DRIFT
  |
  v
CONTINUOUS TRAINING


